# 03 — Model Training
**Global Temperature Forecasting Using Multimodal Machine Learning**  
Choudhary & Kulkarni (2026), *Climatic Change* (Springer)  

Reproduces **§5** of the paper — trains all five deep learning models:

| Model | Key Architecture | Params |
|-------|-----------------|--------|
| LSTM | 128→128→64→1 + 20% dropout | ~130K |
| Bi-LSTM | Bidir(128)+Bidir(64)→1 | ~180K |
| GRU | 128→64→32→1 | ~110K |
| Transformer | 4-head×2 blocks + GAP→64→1 | ~85K |
| **CNN-LSTM** | Conv1D(64,32)+LSTM(64,32)→1 | ~95K |

**Unified training protocol (Table 3):**
- Loss: Huber (δ=1.0) — robust to El Niño outliers
- Optimiser: Adam (lr=1e-3) + ReduceLROnPlateau
- Input: 36-month × 27-feature tensor
- Split: Train 1880–2017 / Val 2017–2018 / Test 2018–2025

In [ ]:
# ── Cell 1: Setup ────────────────────────────────────────────────────────────
# !pip install -q tensorflow xgboost lightgbm scikit-learn statsmodels
import sys, os, time, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from pathlib import Path

from preprocessing import load_all
from feature_engineering import prepare_all
from train_cnn_lstm import build_cnn_lstm, CFG as CNN_CFG
from train_transformer import build_transformer, CFG as TF_CFG
from utils import set_global_seed, apply_style, ensure_dirs

set_global_seed(42)
apply_style()
MODELS_DIR = Path('../models'); MODELS_DIR.mkdir(exist_ok=True)
FIG_DIR    = Path('../results/figures'); FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f'TensorFlow: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

In [ ]:
# ── Cell 2: Load data & build feature sequences ──────────────────────────────
raw  = load_all(use_cache=True)
data = prepare_all(raw, seq_len=36)

X_train, y_train = data['seq_train']
X_val,   y_val   = data['seq_val']
X_test,  y_test  = data['seq_test']
scaler = data['scaler']
n_features = X_train.shape[2]

print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')
print(f'Feature count: {n_features}')

In [ ]:
# ── Cell 3: Define training helper ───────────────────────────────────────────
CFG = {'epochs':120, 'batch_size':32, 'patience':15, 'lr_patience':8}

def train_model(model, name, X_tr, y_tr, X_val, y_val, cfg=CFG):
    ckpt = str(MODELS_DIR / f'{name.lower().replace("-","_").replace(" ","_")}_best.keras')
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=cfg['patience'],
                      restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=cfg['lr_patience'], min_lr=1e-7, verbose=0),
        ModelCheckpoint(ckpt, monitor='val_loss', save_best_only=True, verbose=0),
    ]
    t0 = time.time()
    hist = model.fit(X_tr, y_tr, validation_data=(X_val, y_val),
                     epochs=cfg['epochs'], batch_size=cfg['batch_size'],
                     callbacks=callbacks, verbose=0)
    elapsed = int(time.time() - t0)
    best_val = min(hist.history['val_loss'])
    epochs_done = len(hist.history['loss'])
    print(f'  [{name}] {epochs_done} epochs | best val_loss={best_val:.5f} | {elapsed}s → {ckpt}')
    return hist.history

In [ ]:
# ── Cell 4: Build all five DL models ─────────────────────────────────────────
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (LSTM, GRU, Dense, Dropout, Conv1D,
    MaxPooling1D, Input, Bidirectional, BatchNormalization,
    MultiHeadAttention, LayerNormalization, Add, GlobalAveragePooling1D)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

SEQ, FEAT = 36, n_features
D, LR = 0.25, 1e-3

def lstm_model():
    m = Sequential([Input((SEQ,FEAT)), LSTM(128,return_sequences=True,kernel_regularizer=l2(1e-4)),
        BatchNormalization(), Dropout(D), LSTM(64,return_sequences=True), Dropout(D),
        LSTM(32), Dense(32,'relu'), Dropout(D/2), Dense(1)], name='LSTM')
    m.compile(Adam(LR),'huber',['mae']); return m

def bilstm_model():
    m = Sequential([Input((SEQ,FEAT)), Bidirectional(LSTM(128,return_sequences=True)),
        BatchNormalization(), Dropout(D), Bidirectional(LSTM(64)),
        Dense(64,'relu'), Dropout(D), Dense(1)], name='BiLSTM')
    m.compile(Adam(LR),'huber',['mae']); return m

def gru_model():
    m = Sequential([Input((SEQ,FEAT)), GRU(128,return_sequences=True,kernel_regularizer=l2(1e-4)),
        BatchNormalization(), Dropout(D), GRU(64), Dense(32,'relu'), Dense(1)], name='GRU')
    m.compile(Adam(LR),'huber',['mae']); return m

cnn_lstm   = build_cnn_lstm(SEQ, FEAT)
transformer = build_transformer(SEQ, FEAT)
models = {'LSTM':lstm_model(),'Bi-LSTM':bilstm_model(),'GRU':gru_model(),
          'Transformer':transformer, 'CNN-LSTM':cnn_lstm}

for name, m in models.items():
    print(f'{name:<14}: {m.count_params():>8,} parameters')

In [ ]:
# ── Cell 5: Train all models ─────────────────────────────────────────────────
print('Training all 5 models (this may take 10–30 min on CPU, 3–8 min on GPU):\n')
histories = {}
for name, model in models.items():
    histories[name] = train_model(model, name, X_train, y_train, X_val, y_val)
print('\n✓ All models trained.')

In [ ]:
# ── Cell 6: Training curves (Figure 7) ──────────────────────────────────────
from evaluation import plot_training_curves
fig = plot_training_curves(histories, save_as=None)
fig.savefig(FIG_DIR/'fig7_training_curves.png', dpi=300, bbox_inches='tight')
plt.show()
print('Figure 7 saved.')

In [ ]:
# ── Cell 7: Train ML baselines ───────────────────────────────────────────────
from train_baselines import train_and_evaluate
baseline_results = train_and_evaluate(seed=42)
print('\nBaseline results:')
display(baseline_results)